# Delaunay Triangulation

In this exercise, we will do Delaunay triangulation by means of the the edge-flip algorithm.


In [1]:
from pygel3d import hmesh, spatial, jupyter_display as jd

import numpy as np
from queue import Queue
import plotly.offline as py
import plotly.graph_objs as go

array = np.array
jd.set_export_mode(True)

### Delaunay edge

The following function needs to check whether a given half-edge fullfills the Delaunay property. This can be done by means of the in-circle predicate. In practie, we need to build a 4x4 matrix and check its determinant.

In [ ]:
def delaunay_edge(m, h):
    if m.is_halfedge_at_boundary(h):
        return True

    pos = m.positions() 
    fid = m.incident_face(h)
    verts = m.circulate_face(fid, mode="v")

    oppisite_h = m.opposite_halfedge(h)
    next_h = m.next_halfedge(oppisite_h)
    p4_vid = m.incident_vertex(next_h)
    p4 = pos[p4_vid]
    points = [pos[v] for v in verts] + [p4]

    a = np.array([
        [x, y, x*x + y*y, 1] 
        for x, y, _ in points
    ])

    return np.linalg.det(a) < 0

### Test function
Code below tests the `delaunay_edge` function

In [29]:
m_test = hmesh.Manifold()
m_test.add_face([[0.0,0.0,0.0],[1.0,0.0,0.0],[1.0,1.0,0.0]])
m_test.add_face([[0.0,0.0,0.0],[1.0,1.0,0.0],[0.45,0.55,0.0]])
hmesh.stitch(m_test)
jd.display(m_test)
for h in m_test.halfedges():
    if not delaunay_edge(m_test, h):
        m_test.flip_edge(h)
        print("flipped")
    print("no flip")
jd.display(m_test)

no flip
no flip
no flip
no flip
flipped
no flip
no flip
no flip
no flip
no flip
no flip


### Barycentric coordinates

For the following, we need to determine the barycentric coordinates of a point on a given face. In the following function 

``m`` is a manifold

``f`` a given face

``p`` are the point coordinates

In [ ]:
def barycentrics(m,f, p):
    # Compute and return the barycentric coordinates



    









    pass


In [ ]:
print("b0 = ", barycentrics(m_test,0,[0.4,0.4,0]))
print("b1 = ", barycentrics(m_test,1,[0.4,0.4,0]))

### Filtering points

The following function reduces the number of poins.

``coords`` ar the coordinates of points to be filtered

``rad`` is a radius

In [ ]:
def create_2d_tree(pts):
    tree = spatial.I3DTree()
    for i in range(len(pts)):
        p = array(pts[i])
        p[2] = 0
        tree.insert(p,i)
    tree.build()
    return tree

def filter_points(coords, rad):
    tree = create_2d_tree(coords)
    new_coords = []
    visited = [False]*len(coords)
    for i in range(0,len(coords)):
        if not visited[i]:
            p = array(coords[i])
            new_coords += [array(p)]
            p[2] = 0
            (K,V) = tree.in_sphere(p,rad)
            for idx in V:
                visited[idx] = True
    return new_coords

## Delaunay triangulation pipeline
Now we are ready to run a complete Delayanay triangulation pipeline. 

### Importing data
The code below imports 3D point data and normalizes coordinates such that the x and y components lie within a unit square.

In [ ]:
f = open("./kote1-sorted.txt")
lines = f.readlines()
coords = []
for l in lines:
    coords += [list(map(float, l.split()))]
point_mat = np.array(coords)
spanx = point_mat[:,0].max() - point_mat[:,0].min()
spany = point_mat[:,1].max() - point_mat[:,1].min()
span = max(spanx,spany)
coords -= np.array([point_mat[:,0].min(),point_mat[:,1].min(),point_mat[:,2].min()])
coords *= np.array([1.,1.,5.0])/span

coords = filter_points(coords, 0.001)

### Adding points
You should now add the points in ``coords`` to the manifold ``m`` one by one.

In [ ]:
m = hmesh.Manifold()

# One big "helper triangle" to encompass all normalized points
m.add_face([[-1.,0.,0.],[1.,0.,0.],[1.,2.,0.]])

for c in coords:
    print("Inserting point : ", c)
    # Below, insert algorithmic part of Delaunay triangulation.
    # The code should:
    # - find the appropriate triangle containing c and split it, inserting c
    # - Flip all edges which become not locally Delaunay in the process.
       
# Removing the "helper triangle"
m.remove_vertex(0)
m.remove_vertex(1)
m.remove_vertex(2)

# Removing caps at edge of mesh
for iter in range(15):
    avg_len = hmesh.average_edge_length(m)
    for h in m.halfedges():
        if m.is_halfedge_at_boundary(h) and m.halfedge_in_use(h):
            if m.edge_length(h)>2.5 * avg_len:
                m.remove_edge(h)
m.cleanup()

print("Checking that all edges are Delaunay...")
we_are_good = True
for h in m.halfedges():
    if not delaunay_edge(m,h):
        print("Found a non-Delaunay edge!")
        we_are_good = False
if we_are_good:
    print("We are good!")

In [ ]:
jd.display(m)

### Questions

- Discuss the difference between the notions of locally Delaunay and globally Delaunay
- What is the most efficient Delaunay triangulation algorithm?
- What is most important to the efficiency of this Delaunay triangulation algorithm?
- Why is it a problem if four points share a common circumcircle? What would this algorithm do?

**Diffrence between locally Delaunay and Globally Delaunay**


**Most efficient Delaunay aligorithm**




**Most important to the efficiency of this Delaunay triangulation algorithm**


**Why is it a problem if four points share a common circumcircle**